# LAB 2 for CS4243: Perspective-Aware Virtual Advertisement Overlay
## Notebook 3 — Occlusion handling and compositing
> **Introduction.** This notebook develops **Stage 3 (Composite)** in `overlaylab/compositing.py`. Hysteresis thresholds
> and the minimum component area are chosen on the three **validation** masks of the common clip only
> (`select_difference_thresholds`), frozen, and reported on the three **test** masks of the common clip, on the mask
> frames of the combined and viewpoint clips (section 6) and later on your personal audit masks. The cells marked
> **REQUIRED** must be completed.

## 1. Imports, the six mask frames, and their homographies
The ground-truth masks may be used only to *select thresholds* (validation) and to *evaluate* (test). The homography of
each mask frame comes from your own `localise_board`, never from the ground-truth corners.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import cv2

from overlaylab import data, metrics, visualize as viz
from overlaylab import supplied as sp
from overlaylab.config import notebook_config, replace
from overlaylab.localisation import localise_board, artwork_to_frame
from overlaylab.compositing import hysteresis, compensated_difference, occlusion_mask, select_difference_thresholds, composite
from overlaylab.geometry import warp_artwork
import json

from overlaylab.config import PipelineConfig
threshold_path = ROOT / "outputs" / "thresholds.json"
assert threshold_path.exists(), "run Notebook 2 first to freeze Stage 1 and 2"
config = PipelineConfig.from_json(threshold_path.read_text())
config.resize_width = data.NOTEBOOK_WIDTH
# min_area in the saved file is at original-video resolution.
config.occlusion.min_area = max(1, int(round(config.occlusion.min_area * (data.NOTEBOOK_WIDTH / sp.video_info(data.COMMON_CLIP)["width"]) ** 2)))
reference_full = data.load_reference()
width = data.NOTEBOOK_WIDTH; factor = width / sp.video_info(data.COMMON_CLIP)["width"]
reference = sp.resize_width(reference_full, int(round(reference_full.shape[1] * factor)))
config.localisation.warp.reference_shape = reference.shape[:2]
artwork = data.load_artwork("ad")
masks_val = data.load_gt_masks("validation", width=width); masks_test = data.load_gt_masks("test", width=width)
mask_frames = {}
for i, f in sp.iter_video(data.COMMON_CLIP, step=1, width=width, start=min(masks_val | masks_test), stop=max(masks_val | masks_test) + 1):
    if i in masks_val or i in masks_test:
        mask_frames[i] = f
homs = {i: localise_board(reference, mask_frames[i], config.localisation)[0] for i in mask_frames}
assert all(homs[i] is not None for i in masks_val), "validation localisation failed; investigate Stage 1 on validation data before threshold search"
print("validation masks:", sorted(masks_val), " test masks:", sorted(masks_test), " localised:", [i for i in homs if homs[i] is not None])

## 2. The motion-compensated difference
Warp the reference into the frame, fit a global gain and offset on the best-agreeing pixels, and look at the difference
map. Illumination change alone should *not* light up; the card crossing the poster should.

In [ ]:
i = sorted(masks_val)[0]
warped_ref, cov = warp_artwork(reference, homs[i], mask_frames[i].shape)
diff, (gain, offset) = compensated_difference(mask_frames[i], warped_ref, cov, config.occlusion)
print(f"frame {i}: fitted gain {gain:.3f}, offset {offset:.3f}")
plt.figure(figsize=(15, 4)); plt.imshow(viz.montage([viz.label(mask_frames[i], "frame"), viz.label(warped_ref, "warped reference"), viz.label(viz.heatmap(diff, 0.6), "compensated |difference|"), viz.label(viz.overlay_mask(mask_frames[i], masks_val[i]), "ground-truth occlusion")], cols=4)); plt.axis("off"); plt.show()
plt.figure(figsize=(6, 3)); plt.hist(diff[cov], bins=60, log=True); plt.axvline(config.occlusion.weak, color="orange", ls="--"); plt.axvline(config.occlusion.strong, color="red", ls="--"); plt.xlabel("compensated difference inside coverage"); plt.show()

## 3. Select the thresholds on the validation masks (REQUIRED)
`select_difference_thresholds` must search strong / weak / minimum-area settings on the **validation** masks only and
return the frozen configuration. Record the frozen values in `outputs/thresholds.json`. The minimum area is a pixel
count, so it is stored at full resolution (scaled by `1 / factor**2`); `run_pipeline` rescales it when you pass `--width`.

In [ ]:
# YOUR CODE HERE — call select_difference_thresholds on the validation frames and inspect the search log
val_inputs = [(mask_frames[i], homs[i], reference) for i in sorted(masks_val)]
frozen_occ, log = select_difference_thresholds(val_inputs, [masks_val[i] for i in sorted(masks_val)], config.occlusion)
best = sorted(log, key=lambda r: -r["mean_iou"])[:5]
print("top settings on validation:"); [print("  ", r) for r in best]
config.occlusion = frozen_occ
from overlaylab.config import PipelineConfig
path = ROOT / "outputs" / "thresholds.json"
merged = PipelineConfig.from_json(path.read_text()) if path.exists() else PipelineConfig()   # keep Notebook 2's decisions
merged.occlusion = replace(frozen_occ, min_area=int(round(frozen_occ.min_area / factor ** 2)))   # min_area is stored at full resolution
path.write_text(json.dumps(merged.thresholds(), indent=2)); print("frozen (min_area at full resolution):", merged.thresholds()["occlusion"])

### (Our Code) Native-resolution validation and threshold search

Use full **1280 × 720** common frames and the full-size reference. Only validation masks **96, 192 and 272** select Stage 3 settings. Homographies come from the frozen Stage 1 localiser, with RANSAC seed **0**, rather than ground-truth corners. Stage 1 and Stage 2 remain fixed.

Start from `OcclusionConfig()` on each run and apply the supplied strong/weak/minimum-area search at default opening radius 2. Maximise mean validation IoU and apply the documented central-plateau tie-break. The full search log is available as `native_search_log` during execution. The following opening-radius cell completes selection and saves the final configuration.


In [ ]:
from copy import deepcopy
from overlaylab.config import OcclusionConfig

native_config = PipelineConfig.from_json(threshold_path.read_text())
native_config.resize_width = None
native_config.occlusion = OcclusionConfig()
native_reference = data.load_reference()
native_config.localisation.warp.reference_shape = native_reference.shape[:2]
native_masks_val = data.load_gt_masks("validation")
native_val_frames = {f: image for f, image in sp.iter_video(
    data.COMMON_CLIP, start=min(native_masks_val), stop=max(native_masks_val) + 1)
    if f in native_masks_val}
native_val_homs = {f: localise_board(native_reference, image, native_config.localisation)[0]
                   for f, image in native_val_frames.items()}
native_val_inputs = [(native_val_frames[f], native_val_homs[f], native_reference)
                     for f in sorted(native_masks_val)]
native_selected, native_search_log = select_difference_thresholds(
    native_val_inputs, [native_masks_val[f] for f in sorted(native_masks_val)], native_config.occlusion)
native_best_iou = max(row["mean_iou"] for row in native_search_log)
native_tied = [row for row in native_search_log
               if np.isclose(row["mean_iou"], native_best_iou, rtol=0, atol=1e-12)]
print("Native common validation: 1280 × 720, masks", sorted(native_masks_val))
print("Search candidates:", len(native_search_log), "best mean IoU:", native_best_iou)
print("Tied strong range:", min(row["strong"] for row in native_tied),
      "to", max(row["strong"] for row in native_tied))
print("Selected at radius 2:", native_selected)

### (Our Code) Opening-radius comparison and freeze

Compare radius 2 with radius 0 on the same native validation masks, holding the selected strong/weak thresholds and minimum area fixed. Choose the higher mean validation IoU, retaining radius 2 on an exact tie. This is a conditional opening comparison, not a joint radius/threshold search.

The table reports mean IoU, F1, boundary F1 at **2 native pixels**, direct alpha leakage and false-positive pixels. Leakage counts annotated foreground pixels with alpha > 0.5 and differs from the finished-video pixel-change proxy.

Save the selected Stage 3 settings at the end of this cell, preserving Stage 1, Stage 2 and compositing settings. Minimum area is already in original-video pixels. This native freeze supersedes the starter preview configuration. Subsequent starter cells retain their separate 640-pixel preview configuration, while section 6.1 evaluates native settings without retuning.


In [ ]:
native_val_cache = {}
for f in sorted(native_masks_val):
    warped_ref, cov = warp_artwork(native_reference, native_val_homs[f], native_val_frames[f].shape)
    dmap, fit = compensated_difference(native_val_frames[f], warped_ref, cov, native_selected)
    Ha = artwork_to_frame(artwork, native_val_homs[f], native_config.localisation)
    warped_ad, ad_cov = warp_artwork(artwork, Ha, native_val_frames[f].shape)
    native_val_cache[f] = dict(difference=dmap, coverage=cov, fit=fit, artwork=warped_ad, ad_coverage=ad_cov)
    print(f"frame {f}: gain {fit[0]:.6f}, offset {fit[1]:.6f}")
f = sorted(native_masks_val)[0]
plt.figure(figsize=(15, 4))
plt.imshow(viz.montage([viz.label(native_val_frames[f], f"native validation {f}"),
    viz.label(viz.heatmap(native_val_cache[f]["difference"], 0.6), "compensated colour difference"),
    viz.label(viz.overlay_mask(native_val_frames[f], native_masks_val[f]), "validation ground truth")], cols=3))
plt.axis("off"); plt.show()

def native_mask_measurements(f, occ_config):
    predicted, difference, components = occlusion_mask(
        native_val_frames[f], native_val_homs[f], native_reference, occ_config)
    cached = native_val_cache[f]
    _, alpha = composite(native_val_frames[f], cached["artwork"], cached["ad_coverage"],
                         predicted, native_config.composite)
    target = native_masks_val[f]
    return dict(frame=f, iou=metrics.mask_iou(predicted, target), f1=metrics.mask_f1(predicted, target),
                boundary_f1=metrics.boundary_f1(predicted, target), leakage=metrics.leakage(alpha, target),
                false_positive_px=int((predicted & ~target).sum()), mask_px=int(predicted.sum()))

native_opening_rows = []
for radius in (2, 0):
    measurements = [native_mask_measurements(f, replace(native_selected, open_radius=radius))
                    for f in sorted(native_masks_val)]
    row = dict(open_radius=radius, per_frame=measurements)
    row.update({name: float(np.mean([m[name] for m in measurements]))
                for name in ("iou", "f1", "boundary_f1", "leakage", "false_positive_px")})
    native_opening_rows.append(row)
print("radius    mean IoU    mean F1    boundary F1    leakage    mean false-positive px")
for row in native_opening_rows:
    print(f"{row['open_radius']:6d} {row['iou']:11.6f} {row['f1']:10.6f} "
          f"{row['boundary_f1']:14.6f} {row['leakage']:10.6f} {row['false_positive_px']:25.1f}")
native_radius_choice = max(native_opening_rows, key=lambda row: row["iou"])["open_radius"]
native_occ = replace(native_selected, open_radius=native_radius_choice)
print("Selected opening radius:", native_radius_choice)



native_config.occlusion = native_occ
native_frozen = PipelineConfig.from_json(threshold_path.read_text())
native_frozen.occlusion = deepcopy(native_occ)
threshold_path.write_text(json.dumps(native_frozen.thresholds(), indent=2) + "\n")
print("Final native Stage 3 settings:", json.dumps(native_frozen.thresholds()["occlusion"], indent=2))
print("Compositing settings:", native_frozen.thresholds()["composite"])
print("Localisation success tolerance:", native_frozen.localisation.success_threshold,
      "RANSAC seed:", native_frozen.localisation.ransac.seed)
print("Validation search grid maxima by strong threshold:")
for strong in sorted({r["strong"] for r in native_search_log}):
    print(f"strong {strong:.3f}: best mean IoU {max(r['mean_iou'] for r in native_search_log if r['strong'] == strong):.8f}")

### (Our Code) Thin-occluder survival under the selected opening

Use an ideal vertical pole of width 1–9 pixels, with a strong top segment and a weak lower segment. Its difference values are above the frozen strong and weak thresholds respectively. The pole is 400 pixels long so minimum area does not confound the tested widths. Pass this difference map through the same hysteresis, disc opening and component filtering as `occlusion_mask`, comparing radius 0 with the selected radius. Also disable hysteresis to measure how much of the weak pole is lost.

This controlled demonstration isolates mask cleaning. It does not claim that real, blurred or board-coloured poles produce the same difference signal. Survival means that at least one pole component remains after area filtering. The mask-width table and the weak-pole recall provide evidence for Question 4.

In [ ]:
probe_rows = []
probe_height, probe_width = 440, 80
for pole_width in range(1, 10):
    target = np.zeros((probe_height, probe_width), dtype=bool)
    left = (probe_width - pole_width) // 2
    target[20:420, left:left + pole_width] = True
    signal = np.zeros(target.shape, dtype=np.float32)
    signal[target] = (native_occ.weak + native_occ.strong) / 2
    signal[20:100, left:left + pole_width] = native_occ.strong + 0.05
    for radius in dict.fromkeys((0, native_occ.open_radius)):
        for use_hysteresis in (True, False):
            seeds = signal >= native_occ.strong
            weak = signal >= (native_occ.weak if use_hysteresis else native_occ.strong)
            predicted = hysteresis(seeds, weak, native_occ.connectivity)
            if radius:
                yy, xx = np.mgrid[-radius:radius + 1, -radius:radius + 1]
                disc = (xx * xx + yy * yy <= radius * radius).astype(np.uint8)
                predicted = cv2.morphologyEx(predicted.astype(np.uint8), cv2.MORPH_OPEN, disc).astype(bool)
            count, labels, stats, _ = cv2.connectedComponentsWithStats(
                predicted.astype(np.uint8), connectivity=native_occ.connectivity)
            kept = [k for k in range(1, count) if stats[k, cv2.CC_STAT_AREA] >= native_occ.min_area]
            predicted = np.isin(labels, kept)
            weak_pole = target.copy(); weak_pole[:100] = False
            probe_rows.append(dict(width=pole_width, radius=radius, hysteresis=use_hysteresis,
                remaining_px=int(predicted.sum()), iou=metrics.mask_iou(predicted, target),
                weak_recall=float((predicted & weak_pole).sum() / weak_pole.sum())))
print("width  selected-radius px  radius-0 px  selected weak recall  no-hysteresis weak recall")
for pole_width in range(1, 10):
    selected = next(r for r in probe_rows if r["width"] == pole_width and r["radius"] == native_occ.open_radius and r["hysteresis"])
    zero = next(r for r in probe_rows if r["width"] == pole_width and r["radius"] == 0 and r["hysteresis"])
    no_hyst = next(r for r in probe_rows if r["width"] == pole_width and r["radius"] == native_occ.open_radius and not r["hysteresis"])
    print(f"{pole_width:5d} {selected['remaining_px']:19d} {zero['remaining_px']:12d} "
          f"{selected['weak_recall']:21.6f} {no_hyst['weak_recall']:26.6f}")
thinnest_pole = min(r["width"] for r in probe_rows if r["radius"] == native_occ.open_radius
                    and r["hysteresis"] and r["remaining_px"] > 0)
print("Thinnest surviving ideal pole:", thinnest_pole, "native pixels at radius", native_occ.open_radius)

## 4. Report on the test masks against the no-occlusion baseline (REQUIRED)
IoU, F1, foreground leakage and boundary F1 per test frame, and the same numbers for a baseline that paints the whole board.

In [ ]:
# YOUR CODE HERE — evaluate on the test masks with the frozen configuration; compare with the baseline
print(f"{'frame':>5s} {'IoU':>6s} {'F1':>6s} {'boundary':>9s} {'leakage':>8s} {'baseline leakage':>17s}")
rows = []
for i in sorted(masks_test):
    if homs[i] is None:
        print(f"{i:5d}: localisation failed; report as hidden, do not replace with ground truth")
        continue
    m, dmap, comp = occlusion_mask(mask_frames[i], homs[i], reference, config.occlusion)
    Ha = artwork_to_frame(artwork, homs[i], config.localisation); warped, cov = warp_artwork(artwork, Ha, mask_frames[i].shape)
    out, alpha = composite(mask_frames[i], warped, cov, m, config.composite)
    base, alpha_base = composite(mask_frames[i], warped, cov, np.zeros_like(m), config.composite)
    rows.append((i, metrics.mask_iou(m, masks_test[i]), metrics.mask_f1(m, masks_test[i]), metrics.boundary_f1(m, masks_test[i]),
                 metrics.leakage(alpha, masks_test[i]), metrics.leakage(alpha_base, masks_test[i])))
    print(f"{i:5d} {rows[-1][1]:6.3f} {rows[-1][2]:6.3f} {rows[-1][3]:9.3f} {rows[-1][4]:8.3f} {rows[-1][5]:17.3f}")
i = next((i for i in sorted(masks_test) if homs[i] is not None), None)
assert i is not None, "all test localisations failed; report that failure and analyse Stage 1 evidence"
m, dmap, comp = occlusion_mask(mask_frames[i], homs[i], reference, config.occlusion)
Ha = artwork_to_frame(artwork, homs[i], config.localisation); warped, cov = warp_artwork(artwork, Ha, mask_frames[i].shape)
out, alpha = composite(mask_frames[i], warped, cov, m, config.composite); base, _ = composite(mask_frames[i], warped, cov, np.zeros_like(m), config.composite)
plt.figure(figsize=(15, 6)); plt.imshow(viz.montage([viz.label(mask_frames[i], "original"), viz.label(viz.heatmap(dmap, 0.6), "difference"), viz.label(viz.overlay_mask(mask_frames[i], m), "occlusion mask O"), viz.label(base, "without occlusion handling"), viz.label(out, "with occlusion handling"), viz.label(viz.heatmap(alpha, 1.0), "alpha")], cols=3)); plt.axis("off"); plt.show()
print("component statistics [label, x, y, w, h, area]:"); print(comp)

### (Our Code) Complete no-occlusion baseline metrics

Complete §4's baseline comparison on the same **640-pixel common test frames**. An empty foreground mask disables occlusion handling, so the baseline paints the board wherever artwork coverage and feathering permit. Compute its mask IoU, F1 and boundary F1 against the annotated foreground, and reuse the per-frame baseline leakage already measured in §4. This supplements the starter output without repeating its compositing or figures, changing settings or writing files.


In [ ]:
stage3_baseline_rows = []
print(f"{'frame':>5s} {'baseline IoU':>12s} {'baseline F1':>11s} "
      f"{'baseline boundary F1':>20s} {'baseline leakage':>17s}")
for stage3_test_row in rows:
    stage3_frame = stage3_test_row[0]
    stage3_baseline_mask = np.zeros_like(masks_test[stage3_frame], dtype=bool)
    stage3_baseline_row = dict(
        frame=stage3_frame,
        iou=metrics.mask_iou(stage3_baseline_mask, masks_test[stage3_frame]),
        f1=metrics.mask_f1(stage3_baseline_mask, masks_test[stage3_frame]),
        boundary_f1=metrics.boundary_f1(stage3_baseline_mask, masks_test[stage3_frame]),
        leakage=stage3_test_row[5])
    stage3_baseline_rows.append(stage3_baseline_row)
    print(f"{stage3_frame:5d} {stage3_baseline_row['iou']:12.3f} "
          f"{stage3_baseline_row['f1']:11.3f} {stage3_baseline_row['boundary_f1']:20.3f} "
          f"{stage3_baseline_row['leakage']:17.3f}")


## 5. What the thresholds and the hysteresis step do (**Question 4**)
Sweep the strong threshold and the minimum area on one validation frame and watch the mask and the leakage change; do not use test results to change the frozen settings; disable
hysteresis (weak = strong) to see what happens along the card's edge, where the difference fades through the motion blur.

In [ ]:
i = sorted(masks_val)[1]
Ha = artwork_to_frame(artwork, homs[i], config.localisation); warped, cov = warp_artwork(artwork, Ha, mask_frames[i].shape)
print(f"{'strong':>6s} {'weak':>5s} {'min_area':>8s} {'IoU':>6s} {'leakage':>8s} {'mask px':>8s}")
for strong, weak, area in [(config.occlusion.strong, config.occlusion.weak, config.occlusion.min_area),
                           (config.occlusion.strong * 1.5, config.occlusion.weak, config.occlusion.min_area),
                           (config.occlusion.strong, config.occlusion.weak, config.occlusion.min_area * 6),
                           (config.occlusion.strong, config.occlusion.strong, config.occlusion.min_area)]:
    m, _, _ = occlusion_mask(mask_frames[i], homs[i], reference, replace(config.occlusion, strong=strong, weak=weak, min_area=area))
    _, alpha = composite(mask_frames[i], warped, cov, m, config.composite)
    print(f"{strong:6.2f} {weak:5.2f} {area:8d} {metrics.mask_iou(m, masks_val[i]):6.3f} {metrics.leakage(alpha, masks_val[i]):8.3f} {int(m.sum()):8d}")

## 6. EVALUATION-ONLY GT-FALLBACK — the combined and viewpoint clips with the frozen thresholds (REQUIRED)
The Stage 3 stress clips, `data.stress_clips(3)`. `manifests/diagnostic_masks.csv` lists the mask frames of the
**combined** clip (the blue card crosses the poster while the camera rotates) and of the **viewpoint** clip (three
strong-keystone frames with an *empty* mask: nothing is in front of the board). With the configuration frozen above:

* **combined**: mask IoU / F1 / leakage on the six mask frames against the no-occlusion baseline, as in section 4.
* **viewpoint**: the **false-occlusion fraction** = fraction of the (visible) board that the mask wrongly removes when
  nothing occludes it; the no-occlusion baseline is 0 % false occlusion by construction. Here the homography is always
  available (frozen or GT-fallback), so the overlay is present on every mask frame and the number is a mean over all
  three mask frames. `overlaylab.evaluate` estimates an output-pixel-change proxy on a *finished video*, only on mask frames where
  it detects the artwork in the output: frames where the overlay is absent (hidden by the monitor, tracker lost) are
  listed as `overlay_absent_frames` and excluded, and the printed line says how many frames the mean is based on. The proxy also includes feathering, resampling and similar artwork colours, so it need not equal the direct mask fraction. Always
  quote the frame count with the number. The full-resolution Stage 3 table comes from `python -m overlaylab.report`.

The homography of each frame comes from your `localise_board` with the frozen configuration. When it fails or exceeds the frozen corner-error threshold on a mask frame
(the card covers most of the poster on some combined-clip frames), the cell falls back to the manifest corners through
`cv2.getPerspectiveTransform` **for the mask evaluation only** and labels the row `GT-fallback`; say so in the report and
never use that fallback anywhere else. **Question 4** asks why the viewpoint clip produces false occlusion under strong
perspective even when the homography is correct.

In [ ]:
ref_rect = sp.rect_corners(reference.shape[1], reference.shape[0]).astype(np.float32)
print(f"{'clip':>9s} {'frame':>5s} {'H':>11s} {'IoU':>6s} {'F1':>6s} {'leakage':>8s} {'base leak':>9s} {'false occ':>9s} {'coverage':>8s}")
stress_rows = {}
for level, cid in [(name, data.stress_clips(3)[name]) for name in ("combined", "viewpoint")]:
    video, _ = data.clip_paths(cid)
    gm = {f: (sp.resize_width(m.astype(np.uint8) * 255, width) > 127) for f, m in data.load_diagnostic_masks(cid).items()}
    gc = data.load_diagnostic_mask_corners(cid, scale=factor)
    fr = {i: f for i, f in sp.iter_video(video, step=1, width=width, start=min(gm), stop=max(gm) + 1) if i in gm}
    for f in sorted(fr):
        H, corners, pairs, inliers, diag = localise_board(reference, fr[f], config.localisation)
        source = "frozen"
        if H is None or not diag["status"] or metrics.corner_error(corners, gc[f]) > config.localisation.success_threshold:
            H = cv2.getPerspectiveTransform(ref_rect, gc[f].astype(np.float32)).astype(np.float64); source = "GT-fallback"
        m, dmap, comp = occlusion_mask(fr[f], H, reference, config.occlusion)
        Ha = artwork_to_frame(artwork, H, config.localisation); warped, cov = warp_artwork(artwork, Ha, fr[f].shape)
        out, alpha = composite(fr[f], warped, cov, m, config.composite)
        base, alpha_base = composite(fr[f], warped, cov, np.zeros_like(m), config.composite)
        board = sp.quad_mask(m.shape, gc[f]); visible = board & ~gm[f]
        row = {"clip": level, "frame": f, "H": source,
               "iou": metrics.mask_iou(m, gm[f]) if gm[f].any() else float("nan"), "f1": metrics.mask_f1(m, gm[f]) if gm[f].any() else float("nan"),
               "leakage": metrics.leakage(alpha, gm[f]), "baseline_leakage": metrics.leakage(alpha_base, gm[f]),
               "false_occlusion": float((m & visible).sum() / max(visible.sum(), 1)), "coverage_visible": float(((alpha > 0.5) & visible).sum() / max(visible.sum(), 1))}
        stress_rows[(level, f)] = row
        g = lambda v, fmt="{:6.3f}": (fmt.format(v) if np.isfinite(v) else "   n/a")
        print(f"{level:>9s} {f:5d} {source:>11s} {g(row['iou'])} {g(row['f1'])} {g(row['leakage'], '{:8.3f}')} {g(row['baseline_leakage'], '{:9.3f}')} {row['false_occlusion']:9.3f} {row['coverage_visible']:8.3f}")
    show = sorted(fr)[len(fr) // 2]; H_show = stress_rows[(level, show)]
    m, dmap, _ = occlusion_mask(fr[show], cv2.getPerspectiveTransform(ref_rect, gc[show].astype(np.float32)).astype(np.float64) if H_show["H"] == "GT-fallback" else localise_board(reference, fr[show], config.localisation)[0], reference, config.occlusion)
    plt.figure(figsize=(15, 4)); plt.imshow(viz.montage([viz.label(viz.draw_quad(fr[show], gc[show]), f"{level} clip, frame {show}"), viz.label(viz.heatmap(dmap, 0.6), "compensated difference"), viz.label(viz.overlay_mask(fr[show], m), f"mask O ({H_show['H']})"), viz.label(viz.overlay_mask(fr[show], gm[show]), "ground-truth mask")], cols=4)); plt.axis("off"); plt.show()
nm = lambda xs: float(np.nanmean(xs)) if np.any(np.isfinite(xs)) else float("nan")   # nan when every entry is nan (the viewpoint clip has no foreground)
for level in ("combined", "viewpoint"):
    rows_l = [r for (lv, _), r in stress_rows.items() if lv == level]
    print(f"{level} clip ({data.stress_clips(3)[level]}): mean IoU {nm([r['iou'] for r in rows_l]):.3f}  mean leakage {nm([r['leakage'] for r in rows_l]):.3f} "
          f"(baseline {nm([r['baseline_leakage'] for r in rows_l]):.3f})  false occlusion {np.mean([r['false_occlusion'] for r in rows_l]):.3f} over {len(rows_l)} frames  coverage {np.mean([r['coverage_visible'] for r in rows_l]):.3f} over {len(rows_l)} frames  "
          f"GT-fallback rows {sum(r['H'] == 'GT-fallback' for r in rows_l)}/{len(rows_l)}")
# YOUR CODE HERE — on the viewpoint clip, relate the false-occlusion pixels to the compensated-difference map: where does the warped
# reference disagree with the frame although nothing is in front of the board (resampling blur, specular highlight, keystone)?

### (Our Code) Section 6 viewpoint difference-map diagnostic — EVALUATION-ONLY GT-FALLBACK

At the supplied notebook resolution, compare each viewpoint frame with its warped reference, compensated colour difference and false-occlusion mask. Use the same frozen-localiser or labelled evaluation-only GT-fallback homography as section 6. Print gain/offset, the mean difference on falsely masked and unmasked visible-board pixels, and the false-occlusion fraction. Crop to the annotated board only for illustration. These diagnostics do not change settings or enter the pipeline.


In [ ]:
viewpoint_id = data.stress_clips(3)["viewpoint"]
viewpoint_video, _ = data.clip_paths(viewpoint_id)
viewpoint_corners = data.load_diagnostic_mask_corners(viewpoint_id, scale=factor)
viewpoint_indices = sorted(viewpoint_corners)
viewpoint_frames = {f: image for f, image in sp.iter_video(
    viewpoint_video, width=width, start=min(viewpoint_indices), stop=max(viewpoint_indices) + 1)
    if f in viewpoint_corners}
viewpoint_diagnostics = []
fig, axes = plt.subplots(len(viewpoint_indices), 4, figsize=(16, 11))
print("frame  H             gain     offset   mean diff masked  mean diff unmasked  false occ")
for ax_row, f in zip(axes, viewpoint_indices):
    image = viewpoint_frames[f]
    source = stress_rows[("viewpoint", f)]["H"]
    if source == "GT-fallback":
        H = cv2.getPerspectiveTransform(ref_rect, viewpoint_corners[f].astype(np.float32)).astype(np.float64)
    else:
        H = localise_board(reference, image, config.localisation)[0]
    warped_ref, coverage = warp_artwork(reference, H, image.shape)
    difference, (gain, offset) = compensated_difference(image, warped_ref, coverage, config.occlusion)
    predicted, _, _ = occlusion_mask(image, H, reference, config.occlusion)
    board = sp.quad_mask(predicted.shape, viewpoint_corners[f])
    false_mask = predicted & board
    unmasked = board & ~predicted
    masked_difference = float(difference[false_mask].mean()) if false_mask.any() else float("nan")
    row = dict(frame=f, H=source, gain=gain, offset=offset,
               masked_difference=masked_difference, unmasked_difference=float(difference[unmasked].mean()),
               false_occlusion=float(false_mask.sum() / board.sum()))
    viewpoint_diagnostics.append(row)
    print(f"{f:5d}  {source:11s}  {gain:7.4f}  {offset:8.4f}  {masked_difference:16.6f}  "
          f"{row['unmasked_difference']:18.6f}  {row['false_occlusion']:9.3%}")
    lo = np.maximum(np.floor(viewpoint_corners[f].min(axis=0) - 8).astype(int), 0)
    hi = np.minimum(np.ceil(viewpoint_corners[f].max(axis=0) + 9).astype(int), [image.shape[1], image.shape[0]])
    crop = np.s_[lo[1]:hi[1], lo[0]:hi[0]]
    for ax, picture, title in zip(ax_row,
        [image, warped_ref, viz.heatmap(difference, 0.6), viz.overlay_mask(image, false_mask)],
        ["observed board", "warped reference", "compensated difference", "false-occlusion locations"]):
        ax.imshow(picture[crop])
        ax.set_title(f"{f}: {title}\n{source}")
        ax.axis("off")
plt.tight_layout(); plt.show()


### (Our Code) 6.1 Native-resolution report evidence — EVALUATION-ONLY GT-FALLBACK

This single cell gathers fixed-setting native evidence for the report, following the organisation of Notebook 2 section 6.1. It repeats the supplied validation-frame threshold/area/no-hysteresis ablation, evaluates common-test and stress masks, and shows common/combined boundaries and viewpoint difference-map locations. No settings are selected here. The final pipeline provides the official stress tables, while these direct mask and alpha measurements explain behaviour that the finished-video pixel-change proxy does not isolate.

Common-test frames 144, 208 and 448 use frozen-localiser predictions without fallback. On combined and viewpoint mask frames only, retain the supplied section 6 evaluation exception. If the frozen localiser is unavailable or exceeds the frozen **2%** corner-error tolerance, use annotated corners through `cv2.getPerspectiveTransform` and label the row **GT-fallback**. No fallback enters tracking, videos or generated stress tables.

The common and combined rows include IoU, F1, boundary F1 at 2 native pixels and direct alpha leakage against the no-occlusion baseline. Viewpoint masks are empty, so foreground IoU/F1 and leakage are unavailable. Direct false occlusion divides wrongly masked visible-board pixels by annotated visible-board pixels. Direct coverage divides visible-board pixels with alpha > 0.5 by that same visible-board denominator. Quote frame and fallback counts with summaries.

The boundary figure uses common frame 144 and combined frame 216. Red marks false-mask pixels, and cyan marks annotated foreground missed by the mask. Ground truth defines illustration crops and scoring only.

The viewpoint figures here and the dedicated section 6 diagnostic above address the supplied section 6 `YOUR CODE HERE` ask: locate false masks in the compensated-difference map and explain why local appearance can disagree despite adequate corner geometry. The original comment is retained unchanged. A global gain/offset fit cannot undo local resampling blur, view-dependent appearance or residual registration error, and the figures do not isolate those causes from one another.

In [ ]:
native_ablations = []
f = 192
for label, candidate in [
    ("selected", native_occ),
    ("strong × 1.5", replace(native_occ, strong=native_occ.strong * 1.5)),
    ("minimum area × 6", replace(native_occ, min_area=native_occ.min_area * 6)),
    ("no hysteresis", replace(native_occ, weak=native_occ.strong))]:
    measured = native_mask_measurements(f, candidate)
    native_ablations.append(dict(label=label, strong=candidate.strong, weak=candidate.weak,
                                 min_area=candidate.min_area, **measured))
print("Native validation frame 192: ablations")
for row in native_ablations:
    print(f"{row['label']:18s} strong {row['strong']:.3f} weak {row['weak']:.3f} "
          f"area {row['min_area']:4d} IoU {row['iou']:.6f} leakage {row['leakage']:.6f} mask px {row['mask_px']}")

native_masks_test = data.load_gt_masks("test")
native_test_frames = {f: image for f, image in sp.iter_video(
    data.COMMON_CLIP, start=min(native_masks_test), stop=max(native_masks_test) + 1)
    if f in native_masks_test}
native_test_rows = []
native_test_examples = {}
print("frame   IoU       F1        boundary F1   leakage     baseline leakage   H")
for f in sorted(native_masks_test):
    image = native_test_frames[f]
    H, corners, pairs, inliers, diag = localise_board(native_reference, image, native_config.localisation)
    if H is None:
        native_test_rows.append(dict(frame=f, H="unavailable"))
        print(f, "unavailable model, hidden, no GT fallback")
        continue
    predicted, dmap, components = occlusion_mask(image, H, native_reference, native_occ)
    Ha = artwork_to_frame(artwork, H, native_config.localisation)
    warped, cov = warp_artwork(artwork, Ha, image.shape)
    output, alpha = composite(image, warped, cov, predicted, native_config.composite)
    baseline, baseline_alpha = composite(image, warped, cov, np.zeros_like(predicted), native_config.composite)
    target = native_masks_test[f]
    row = dict(frame=f, H="frozen", iou=metrics.mask_iou(predicted, target),
               f1=metrics.mask_f1(predicted, target), boundary_f1=metrics.boundary_f1(predicted, target),
               leakage=metrics.leakage(alpha, target), baseline_leakage=metrics.leakage(baseline_alpha, target))
    native_test_rows.append(row)
    native_test_examples[f] = dict(frame=image, target=target, mask=predicted, difference=dmap,
                                   output=output, baseline=baseline, alpha=alpha, components=components)
    print(f"{f:5d} {row['iou']:.6f}  {row['f1']:.6f}  {row['boundary_f1']:.6f}      "
          f"{row['leakage']:.6f}    {row['baseline_leakage']:.6f}           frozen")
print("Native common test scored:", len(native_test_examples), "/", len(native_masks_test),
      "unavailable:", len(native_masks_test) - len(native_test_examples))
f = 144
example = native_test_examples[f]
plt.figure(figsize=(15, 7))
plt.imshow(viz.montage([viz.label(example["frame"], f"native common test {f}"),
    viz.label(viz.heatmap(example["difference"], 0.6), "difference"),
    viz.label(viz.overlay_mask(example["frame"], example["mask"]), "predicted mask"),
    viz.label(example["baseline"], "no-occlusion baseline"), viz.label(example["output"], "occlusion-aware composite"),
    viz.label(viz.heatmap(example["alpha"], 1.0), "alpha")], cols=3))
plt.axis("off"); plt.show()

native_stress_rows = []
native_stress_examples = {}
native_ref_rect = sp.rect_corners(native_reference.shape[1], native_reference.shape[0]).astype(np.float32)
for level in ("combined", "viewpoint"):
    cid = data.stress_clips(3)[level]
    video, _ = data.clip_paths(cid)
    targets = data.load_diagnostic_masks(cid)
    corners_gt = data.load_diagnostic_mask_corners(cid)
    images = {f: image for f, image in sp.iter_video(video, start=min(targets), stop=max(targets) + 1)
              if f in targets}
    for f in sorted(targets):
        image, target = images[f], targets[f]
        H, corners, pairs, inliers, diag = localise_board(native_reference, image, native_config.localisation)
        source = "frozen"
        error = metrics.corner_error(corners, corners_gt[f]) if H is not None else float("inf")
        if H is None or not diag["status"] or error > native_config.localisation.success_threshold:
            H = cv2.getPerspectiveTransform(native_ref_rect, corners_gt[f].astype(np.float32)).astype(np.float64)
            source = "GT-fallback"
        predicted, dmap, components = occlusion_mask(image, H, native_reference, native_occ)
        Ha = artwork_to_frame(artwork, H, native_config.localisation)
        warped, cov = warp_artwork(artwork, Ha, image.shape)
        output, alpha = composite(image, warped, cov, predicted, native_config.composite)
        baseline, baseline_alpha = composite(image, warped, cov, np.zeros_like(predicted), native_config.composite)
        board = sp.quad_mask(predicted.shape, corners_gt[f])
        visible = board & ~target
        row = dict(clip=level, frame=f, H=source, localiser_error=error,
            iou=metrics.mask_iou(predicted, target) if target.any() else float("nan"),
            f1=metrics.mask_f1(predicted, target) if target.any() else float("nan"),
            boundary_f1=metrics.boundary_f1(predicted, target) if target.any() else float("nan"),
            leakage=metrics.leakage(alpha, target), baseline_leakage=metrics.leakage(baseline_alpha, target),
            false_occlusion=float((predicted & visible).sum() / visible.sum()),
            coverage=float(((alpha > 0.5) & visible).sum() / visible.sum()))
        native_stress_rows.append(row)
        native_stress_examples[(level, f)] = dict(frame=image, target=target, mask=predicted,
            difference=dmap, output=output, baseline=baseline, alpha=alpha, board=board,
            visible=visible, components=components, homography=H)
        print(json.dumps(row))
    group = [r for r in native_stress_rows if r["clip"] == level]
    summary = {key: float(np.mean([r[key] for r in group])) for key in
               (("iou", "f1", "boundary_f1", "leakage", "baseline_leakage") if level == "combined"
                else ("false_occlusion", "coverage"))}
    print(level, "native mask-isolation summary:", summary, "frames:", len(group),
          "GT-fallback:", sum(r["H"] == "GT-fallback" for r in group))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax_row, label, example in [(axes[0], "common 144", native_test_examples[144]),
                              (axes[1], "combined 216", native_stress_examples[("combined", 216)])]:
    yy, xx = np.where(example["target"])
    y0, y1 = max(0, yy.min() - 24), min(example["mask"].shape[0], yy.max() + 25)
    x0, x1 = max(0, xx.min() - 24), min(example["mask"].shape[1], xx.max() + 25)
    crop = np.s_[y0:y1, x0:x1]
    errors = example["frame"].copy()
    errors[example["mask"] & ~example["target"]] = (1, 0, 0)
    errors[example["target"] & ~example["mask"]] = (0, 1, 1)
    for ax, image, title in zip(ax_row,
        [viz.heatmap(example["difference"], 0.6), errors, example["output"]],
        ["compensated difference", "red false mask, cyan missed foreground", "composited boundary"]):
        ax.imshow(image[crop]); ax.set_title(f"{label}: {title}"); ax.axis("off")
native_boundary_figure = fig
plt.tight_layout(); plt.show()

native_viewpoint_diagnostics = []
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for ax_row, row in zip(axes, [r for r in native_stress_rows if r["clip"] == "viewpoint"]):
    f = row["frame"]
    example = native_stress_examples[("viewpoint", f)]
    corners_gt = data.load_diagnostic_mask_corners(data.stress_clips(3)["viewpoint"])[f]
    warped_reference, reference_coverage = warp_artwork(native_reference, example["homography"], example["frame"].shape)
    difference, (gain, offset) = compensated_difference(
        example["frame"], warped_reference, reference_coverage, native_occ)
    false_mask = example["mask"] & example["visible"]
    unmasked = example["visible"] & ~example["mask"]
    native_viewpoint_diagnostics.append(dict(frame=f, H=row["H"], gain=gain, offset=offset,
        mean_difference_masked=float(difference[false_mask].mean()) if false_mask.any() else float("nan"),
        mean_difference_unmasked=float(difference[unmasked].mean()),
        false_mask_px=int(false_mask.sum()), visible_board_px=int(example["visible"].sum())))
    lo = np.maximum(np.floor(corners_gt.min(axis=0) - 8).astype(int), 0)
    hi = np.minimum(np.ceil(corners_gt.max(axis=0) + 9).astype(int),
                    [example["frame"].shape[1], example["frame"].shape[0]])
    crop = np.s_[lo[1]:hi[1], lo[0]:hi[0]]
    for ax, picture, title in zip(ax_row,
        [example["frame"], warped_reference, viz.heatmap(difference, 0.6),
         viz.overlay_mask(example["frame"], false_mask)],
        ["observed board", "warped reference", "compensated difference",
         f"false mask {row['false_occlusion']:.3%}"]):
        ax.imshow(picture[crop]); ax.set_title(f"viewpoint {f}: {title}\n{row['H']}"); ax.axis("off")
    print(f"viewpoint {f}: frozen localiser error {row['localiser_error']:.6f}, "
          f"H source {row['H']}, false-mask pixels {int(false_mask.sum())}, "
          f"visible-board denominator {int(example['visible'].sum())}")
    print("Native viewpoint difference diagnostic:", json.dumps(native_viewpoint_diagnostics[-1]))
native_viewpoint_figure = fig
plt.tight_layout(); plt.show()


### (Our Code) 6.2 Save Stage 3 report evidence

Export the existing native boundary and viewpoint figures without rerunning an experiment. Save the complete validation search, opening comparison, fixed-setting ablations, thin-pole probe and direct mask measurements in `student_files/stage3/evidence.json`. These independent-localiser and labelled GT-fallback diagnostics explain Stage 3 behaviour. The official flow-assisted results remain the generated stress table and finished-video evaluator reports. No settings change here.


In [ ]:
stage3_dir = ROOT / "student_files" / "stage3"
stage3_dir.mkdir(parents=True, exist_ok=True)
for name, figure in [("native_boundaries.png", native_boundary_figure),
                      ("native_viewpoint.png", native_viewpoint_figure)]:
    figure.savefig(stage3_dir / name, dpi=160, bbox_inches="tight", facecolor="white")
    print("Saved", stage3_dir / name)

native_error_rows = []
for label, example in [("common 144", native_test_examples[144]),
                       ("combined 216", native_stress_examples[("combined", 216)])]:
    target, predicted, difference = example["target"], example["mask"], example["difference"]
    false_positive = predicted & ~target
    false_negative = target & ~predicted
    row = dict(example=label, false_positive_px=int(false_positive.sum()),
               false_negative_px=int(false_negative.sum()),
               mean_difference_false_positive=float(difference[false_positive].mean()) if false_positive.any() else None,
               mean_difference_false_negative=float(difference[false_negative].mean()) if false_negative.any() else None,
               components=example["components"].tolist())
    native_error_rows.append(row)
    print("Boundary diagnostic:", json.dumps(row))

def stage3_json_value(value):
    if isinstance(value, dict):
        return {str(key): stage3_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [stage3_json_value(item) for item in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value

stage3_evidence = dict(
    resolution=[1280, 720], method="independent frozen localiser, labelled evaluation-only GT-fallback on stress masks",
    validation_frames=sorted(native_masks_val), test_frames=sorted(native_masks_test),
    frozen_config=native_frozen.thresholds(), validation_search=native_search_log,
    validation_gain_offset={f: native_val_cache[f]["fit"] for f in sorted(native_val_cache)},
    opening_comparison=native_opening_rows, validation_frame_192_ablations=native_ablations,
    thin_pole_probe=probe_rows, thinnest_surviving_pole=thinnest_pole,
    common_test=native_test_rows, stress_masks=native_stress_rows, boundary_errors=native_error_rows,
    viewpoint_difference=native_viewpoint_diagnostics)
(stage3_dir / "evidence.json").write_text(
    json.dumps(stage3_json_value(stage3_evidence), indent=2, allow_nan=False) + "\n")
assert PipelineConfig.from_json(threshold_path.read_text()).thresholds() == native_frozen.thresholds()
print("Saved complete Stage 3 evidence log. Frozen settings unchanged.")


## 7. Optional ablation — `mode="colour"`
A colour-histogram back-projection of the board appearance as an alternative occlusion signal.

In [ ]:
try:
    m_col, d_col, _ = occlusion_mask(mask_frames[i], homs[i], reference, replace(config.occlusion, mode="colour"))
    print("colour mode IoU on frame", i, ":", round(metrics.mask_iou(m_col, (masks_val | masks_test)[i]), 3))
except NotImplementedError:
    print("colour mode not implemented (optional)")

## Reflection prompts
1. Which pixels of the card are missed, and what in the difference map explains it (hint: compare the card's colour with the poster's sky)?
2. Where does the mask leak outside the card, and is that a homography error or a difference-threshold error?
3. The card is a uniform, strongly coloured occluder. Which occluder in your own clip (an arm, a bag, a ball) would be harder, and why?
4. What would a shadow cast on the poster do to the mask, and how would you tell it apart from an occluder?
5. On the viewpoint clip the homography is correct and nothing occludes the board, yet pixels are masked. Which assumption of the differencing fails under strong perspective (**Question 4**)?